In [1]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import json
import plotly.graph_objects as go
from collections import Counter
import nltk
from nltk.corpus import stopwords
import string
import re
import textstat  # для метрик читаемости

# Загрузка данных (Файлы уже есть в директории проекта, там же где и ноутбук)
billie_df = pd.read_csv('billie_eilish_songs.csv')
adele_df = pd.read_csv('adele_songs.csv')

# Добавляем метрики для каждого текста
def calculate_text_metrics(df):
    df['ttr'] = df['lyrics'].apply(calculate_ttr)
    df['flesch_reading'] = df['lyrics'].apply(lambda x: textstat.flesch_reading_ease(str(x)))
    return df

def calculate_ttr(text):
    try:
        text = str(text).lower()
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'[^\w\s]', '', text)
        words = text.split()
        words = [word for word in words if len(word) > 2]
        if not words:
            return 0
        unique_words = set(words)
        return len(unique_words) / len(words)
    except:
        return 0

# Применяем расчет метрик
billie_df = calculate_text_metrics(billie_df)
adele_df = calculate_text_metrics(adele_df)

combined_df = pd.concat([billie_df, adele_df])

# Инициализация Dash app
app = dash.Dash(__name__)

# Улучшенная функция для создания данных для облака слов
def generate_wordcloud_data(text, max_words=100):
    try:
        # Очистка текста
        text = text.lower()
        text = re.sub(r'\[.*?\]', '', text)  # Удаляем текст в квадратных скобках
        text = re.sub(r'[^\w\s]', '', text)  # Удаляем пунктуацию
        words = text.split()  # Простое разделение по пробелам вместо word_tokenize
        
        # Удаляем стоп-слова и короткие слова
        stop_words = set(stopwords.words('english'))
        words = [word for word in words if word not in stop_words and len(word) > 2]
        
        word_freq = Counter(words)
        return word_freq.most_common(max_words)
    except Exception as e:
        print(f"Error in generate_wordcloud_data: {e}")
        return []

# Создание layout
app.layout = html.Div([
    html.H1("Анализ текстов песен Billie Eilish и Adele", style={'textAlign': 'center'}),
    
    dcc.Tabs([
        dcc.Tab(label='Общая статистика', children=[
            html.Div([
                dcc.Dropdown(
                    id='artist-selector',
                    options=[
                        {'label': 'Billie Eilish', 'value': 'Billie Eilish'},
                        {'label': 'Adele', 'value': 'Adele'},
                        {'label': 'Оба исполнителя', 'value': 'Both'}
                    ],
                    value='Both',
                    style={'width': '50%', 'margin': '0 auto'}
                ),
                dcc.Graph(id='word-count-plot'),
                dcc.Graph(id='char-count-plot'),
                dcc.Graph(id='album-stats-plot')
            ])
        ]),
        
        dcc.Tab(label='Анализ слов', children=[
            html.Div([
                dcc.Dropdown(
                    id='wordcloud-artist',
                    options=[
                        {'label': 'Billie Eilish', 'value': 'Billie Eilish'},
                        {'label': 'Adele', 'value': 'Adele'}
                    ],
                    value='Billie Eilish',
                    style={'width': '50%', 'margin': '0 auto'}
                ),
                dcc.Graph(id='wordcloud-plot'),
                dcc.Graph(id='top-words-plot')
            ])
        ]),
        
        dcc.Tab(label='Сравнение', children=[
            html.Div([
                dcc.Graph(id='ttr-comparison'),
                dcc.Graph(id='readability-comparison'),
                dcc.Graph(id='word-count-comparison')
            ])
        ])
    ])
])

# Callbacks для интерактивности
@app.callback(
    [Output('word-count-plot', 'figure'),
     Output('char-count-plot', 'figure'),
     Output('album-stats-plot', 'figure')],
    [Input('artist-selector', 'value')]
)
def update_stats(artist):
    if artist == 'Both':
        df = combined_df
        title_suffix = 'Оба исполнителя'
    else:
        df = combined_df[combined_df['artist'] == artist]
        title_suffix = artist
    
    # График количества слов
    word_fig = px.box(df, x='artist', y='word_count', 
                     title=f'Распределение количества слов ({title_suffix})',
                     color='artist')
    
    # График количества символов
    char_fig = px.box(df, x='artist', y='char_count', 
                     title=f'Распределение количества символов ({title_suffix})',
                     color='artist')
    
    # График статистики по альбомам
    album_counts = df.groupby(['artist', 'album_name']).size().reset_index(name='count')
    album_fig = px.bar(album_counts,
                      x='album_name', y='count', 
                      title=f'Количество песен по альбомам ({title_suffix})',
                      color='artist', barmode='group')
    
    return word_fig, char_fig, album_fig

@app.callback(
    [Output('wordcloud-plot', 'figure'),
     Output('top-words-plot', 'figure')],
    [Input('wordcloud-artist', 'value')]
)
def update_word_analysis(artist):
    if artist == 'Billie Eilish':
        text = ' '.join(billie_df['lyrics'].astype(str))
    else:
        text = ' '.join(adele_df['lyrics'].astype(str))
    
    # Облако слов
    wordcloud_data = generate_wordcloud_data(text)
    wordcloud_df = pd.DataFrame(wordcloud_data, columns=['word', 'count'])
    wordcloud_fig = px.bar(wordcloud_df,
                         x='word', y='count',
                         title=f'Частота слов ({artist})')
    
    # Топ 20 слов
    top_words = wordcloud_data[:20]
    top_words_df = pd.DataFrame(top_words, columns=['word', 'count'])
    top_words_fig = px.bar(top_words_df,
                         x='count', y='word', orientation='h',
                         title=f'Топ 20 слов ({artist})')
    
    return wordcloud_fig, top_words_fig

@app.callback(
    [Output('ttr-comparison', 'figure'),
     Output('readability-comparison', 'figure'),
     Output('word-count-comparison', 'figure')],
    [Input('artist-selector', 'value')]  # Это input не используется, но нужен для callback
)
def update_comparison(_):
    # Сравнение TTR (Type-Token Ratio)
    ttr_fig = px.box(combined_df, x='artist', y='ttr', 
                    title='Сравнение TTR (Type-Token Ratio)',
                    color='artist')
    
    # Сравнение читаемости
    readability_fig = px.box(combined_df, x='artist', y='flesch_reading', 
                           title='Сравнение уровня читаемости (Flesch Reading Ease)',
                           color='artist')
    
    # Сравнение количества слов
    word_count_fig = px.box(combined_df, x='artist', y='word_count',
                           title='Сравнение количества слов в песнях',
                           color='artist')
    
    return ttr_fig, readability_fig, word_count_fig

# Запуск приложения
if __name__ == '__main__':
    app.run(debug=True, port=8052)